# Flux-ratio anomalies from a small-scale perturber

This notebook builds on the quad modelling notebook. We simulate a quadruply imaged quasar whose lens has one extra ingredient: a small **singular isothermal sphere (SIS)** perturber (for example a dark matter subhalo) sitting close to one of the lensed images.

We then do what an observer would do. We fit the image with a smooth **macro model** only (power-law ellipsoid + external shear, with no perturber) and compare the measured quasar image fluxes with the fluxes the macro model predicts.

The perturber shifts the image positions slightly, but the smooth model can absorb such shifts by adjusting its parameters. The magnification of the nearby image changes a lot, though, and the smooth model can't absorb that. It shows up as a **flux-ratio anomaly**. This is the basic idea behind using flux ratios of lensed quasars to detect dark matter substructure.

## How to run this notebook in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amn3142/RESCEU_dm/blob/main/quad_with_sis_perturber.ipynb)

You only need a web browser and a Google account. Nothing is installed on your computer.

1. **Open the notebook in Colab.** Click the **Open in Colab** button above. If you downloaded the `.ipynb` file instead, go to [colab.research.google.com](https://colab.research.google.com) and choose **File → Upload notebook**.
2. **Sign in** with your Google account if asked.
3. *(Optional)* Choose **File → Save a copy in Drive** if you want to keep your changes. Otherwise they are lost when you close the tab.
4. **Run everything** with **Runtime → Run all** (Ctrl+F9, or ⌘+F9 on a Mac).
5. When Colab says *"This notebook was not authored by Google"*, click **Run anyway**.
6. **Wait.** The first code cell installs `lenstronomy`, `emcee` and `tqdm`, which takes about a minute. The model fit takes a couple of minutes and shows a progress bar. A green check next to a cell means it has finished.

**Troubleshooting**
- If Colab asks you to **Restart session**, click it and then choose **Runtime → Run all** again. The install is skipped the second time.
- If you leave the notebook idle for a long time, Colab resets it. Just choose **Run all** again.
- You don't need a GPU. The default runtime is fine.

In [ ]:
# some python imports
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

# make sure lenstronomy is installed, otherwise install the latest pip version
try:
    import lenstronomy
except:
    %pip install lenstronomy emcee tqdm

import lenstronomy.Util.simulation_util as sim_util
import lenstronomy.Util.image_util as image_util
from lenstronomy.Util import param_util
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.PointSource.point_source import PointSource
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver
from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.Data.psf import PSF
from lenstronomy.Workflow.fitting_sequence import FittingSequence
from lenstronomy.Plots.model_plot import ModelPlot

np.random.seed(42)  # fix the noise realization so everyone gets the same result

## The smooth macro model

This is the same setup as in the quad modelling notebook: a power-law elliptical mass profile (`EPL`) with external shear, a Sersic lens light and an elliptical Sersic source that hosts the quasar.

In [ ]:
# data specifics
background_rms = .5  # background noise per pixel
exp_time = 100  # exposure time (arbitrary units, flux per pixel is in units #photons/exp_time unit)
numPix = 100  # cutout pixel size
deltaPix = 0.05  # pixel size in arcsec (area per pixel = deltaPix**2)
fwhm = 0.1  # full width half max of PSF

kwargs_data = sim_util.data_configure_simple(numPix, deltaPix, exp_time, background_rms)
data_class = ImageData(**kwargs_data)
kwargs_psf = {'psf_type': 'GAUSSIAN', 'fwhm': fwhm, 'pixel_size': deltaPix, 'truncation': 5}
psf_class = PSF(**kwargs_psf)

# smooth macro lens model: power-law ellipsoid + external shear
lens_model_list_macro = ['EPL', 'SHEAR']
gamma1, gamma2 = param_util.shear_polar2cartesian(phi=0.1, gamma=0.02)
kwargs_shear = {'gamma1': gamma1, 'gamma2': gamma2}
kwargs_pemd = {'theta_E': 1., 'gamma': 1.96, 'center_x': 0, 'center_y': 0, 'e1': 0.1, 'e2': 0.2}
kwargs_lens_macro = [kwargs_pemd, kwargs_shear]
lens_model_macro = LensModel(lens_model_list=lens_model_list_macro)

# lens light: spherical Sersic profile
lens_light_model_list = ['SERSIC']
kwargs_lens_light = [{'amp': 400, 'R_sersic': 1., 'n_sersic': 2, 'center_x': 0, 'center_y': 0}]
lens_light_model_class = LightModel(light_model_list=lens_light_model_list)

# source light: elliptical Sersic profile, hosting the quasar at its center
source_model_list = ['SERSIC_ELLIPSE']
ra_source, dec_source = 0., 0.1
kwargs_source = [{'amp': 4000., 'R_sersic': .1, 'n_sersic': 3, 'center_x': ra_source,
                  'center_y': dec_source, 'e1': -0.1, 'e2': 0.01}]
source_model_class = LightModel(light_model_list=source_model_list)

# quasar image positions and magnifications for the macro model alone
x_image_macro, y_image_macro = LensEquationSolver(lens_model_macro).find_bright_image(
    ra_source, dec_source, kwargs_lens_macro, numImages=4, min_distance=deltaPix, search_window=numPix * deltaPix)
mag_macro = lens_model_macro.magnification(x_image_macro, y_image_macro, kwargs=kwargs_lens_macro)
# the sign of the magnification is the parity of the image: > 0 for minima, < 0 for saddle points
print('macro model magnifications:', np.round(mag_macro, 2))

## Adding the SIS perturber

We place a small SIS next to one of the images. Its Einstein radius `theta_E_sub` sets its mass. It sits `offset_sub` arcsec further out from the lens center than the image it perturbs.

Images at **saddle points** (negative parity) and at **minima** (positive parity) respond differently to a perturber. By default we perturb the brightest saddle image. Try changing the settings below and re-running the notebook.

In [ ]:
# ---- settings you can play with ----
theta_E_sub = 0.02          # Einstein radius of the SIS perturber in arcsec
offset_sub = 0.08           # distance of the perturber from the lensed image in arcsec
perturbed_image = 'saddle'  # 'saddle' or 'minimum': which type of image to perturb
# -------------------------------------

# pick the brightest image of the requested parity
if perturbed_image == 'saddle':
    candidates = np.where(mag_macro < 0)[0]
else:
    candidates = np.where(mag_macro > 0)[0]
i_pert = candidates[np.argmax(np.abs(mag_macro[candidates]))]

# place the perturber radially outward from that image
r_image = np.hypot(x_image_macro[i_pert], y_image_macro[i_pert])
x_sub = x_image_macro[i_pert] * (1 + offset_sub / r_image)
y_sub = y_image_macro[i_pert] * (1 + offset_sub / r_image)
kwargs_sis = {'theta_E': theta_E_sub, 'center_x': x_sub, 'center_y': y_sub}

# the true lens model = macro model + SIS perturber
lens_model_list_true = lens_model_list_macro + ['SIS']
kwargs_lens_true = kwargs_lens_macro + [kwargs_sis]
lens_model_true = LensModel(lens_model_list=lens_model_list_true)

# a perturber that is too massive or too close to an image can split it into extra images
x_all, _ = LensEquationSolver(lens_model_true).image_position_from_source(
    ra_source, dec_source, kwargs_lens_true, min_distance=0.01, search_window=numPix * deltaPix)
if len(x_all) != 4:
    print(f'WARNING: with these settings the perturber creates {len(x_all)} images instead of 4, and the rest of '
          'this notebook will not be meaningful. Increase offset_sub or decrease theta_E_sub.')

x_image, y_image = LensEquationSolver(lens_model_true).find_bright_image(
    ra_source, dec_source, kwargs_lens_true, numImages=4, min_distance=deltaPix, search_window=numPix * deltaPix)
# put the images in the same order as the macro-model images
order = [np.argmin(np.hypot(x_image - x, y_image - y)) for x, y in zip(x_image_macro, y_image_macro)]
x_image, y_image = x_image[order], y_image[order]
mag_true = lens_model_true.magnification(x_image, y_image, kwargs=kwargs_lens_true)

labels = ['A', 'B', 'C', 'D']

## Simulating the observation

We simulate the image with the true lens model. The quasar fluxes are set by the true magnifications. Unlike the quad modelling notebook, we don't add random microlensing noise to the fluxes, so the only source of a flux anomaly is the SIS.

The right panel shows the noise-free difference between the images with and without the perturber.

Below the figure, a table compares each image's magnification with and without the perturber.

In [ ]:
kwargs_numerics = {'supersampling_factor': 1, 'supersampling_convolution': False}
point_source_list = ['LENSED_POSITION']
point_source_class = PointSource(point_source_type_list=point_source_list, fixed_magnification_list=[False])
quasar_flux = 100  # intrinsic quasar brightness

def simulate(lens_model_list, kwargs_lens, x_img, y_img, mag):
    image_model = ImageModel(data_class, psf_class, LensModel(lens_model_list), source_model_class,
                             lens_light_model_class, point_source_class, kwargs_numerics=kwargs_numerics)
    kwargs_ps = [{'ra_image': x_img, 'dec_image': y_img, 'point_amp': np.abs(mag) * quasar_flux}]
    return image_model.image(kwargs_lens, kwargs_source, kwargs_lens_light, kwargs_ps)

image_true = simulate(lens_model_list_true, kwargs_lens_true, x_image, y_image, mag_true)
image_no_sub = simulate(lens_model_list_macro, kwargs_lens_macro, x_image_macro, y_image_macro, mag_macro)

# add noise to the true image
poisson = image_util.add_poisson(image_true, exp_time=exp_time)
bkg = image_util.add_background(image_true, sigma_bkd=background_rms)
image_sim = image_true + bkg + poisson
kwargs_data['image_data'] = image_sim
data_class.update_data(image_sim)

# pixel coordinates of the images and the perturber, for plotting
x_pix, y_pix = data_class.map_coord2pix(x_image, y_image)
x_sub_pix, y_sub_pix = data_class.map_coord2pix(x_sub, y_sub)

f, axes = plt.subplots(1, 2, figsize=(14, 5.5))
im = axes[0].matshow(np.log10(np.maximum(image_sim, 1e-1)), origin='lower')
axes[0].set_title('simulated observation (log scale)')
f.colorbar(im, ax=axes[0])
diff = image_true - image_no_sub
vmax = np.max(np.abs(diff))
im = axes[1].matshow(diff, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('difference: with $-$ without SIS')
f.colorbar(im, ax=axes[1])
for ax in axes:
    ax.plot(x_sub_pix, y_sub_pix, 'x', color='k', ms=10, mew=2, label='SIS perturber')
    for i in range(4):
        ax.text(x_pix[i] + 4, y_pix[i] + 4, labels[i], color='k', fontsize=14, fontweight='bold')
    ax.xaxis.set_ticks_position('bottom')
axes[0].legend(loc='lower left')
plt.show()

print('perturbed image:', labels[i_pert])
print('image  parity    |mu| macro   |mu| with SIS   change')
for i in range(4):
    parity = 'minimum' if mag_macro[i] > 0 else 'saddle '
    print(f'  {labels[i]}    {parity}     {abs(mag_macro[i]):6.2f}       {abs(mag_true[i]):6.2f}      {abs(mag_true[i]) / abs(mag_macro[i]) - 1:+.0%}')

## Fitting with the smooth macro model only

Now we pretend we don't know about the perturber and fit the image with the smooth macro model (`EPL` + `SHEAR`). As in the quad modelling notebook, the `PROFILE_SHEAR` solver makes the lens model reproduce the four quasar image positions exactly.

The four quasar fluxes are fitted as independent amplitudes (`fixed_magnification_list=[False]`). They are our **measured** fluxes. To keep this quick, we only run the particle swarm optimizer (PSO) and skip the MCMC.

In [ ]:
kwargs_model = {'lens_model_list': lens_model_list_macro,
                'source_light_model_list': source_model_list,
                'lens_light_model_list': lens_light_model_list,
                'point_source_model_list': point_source_list,
                'additional_images_list': [False],
                'fixed_magnification_list': [False],  # each image gets its own, independent flux
                }

kwargs_constraints = {'joint_source_with_point_source': [[0, 0]],
                      'num_point_source_list': [4],
                      'solver_type': 'PROFILE_SHEAR',
                      }

kwargs_likelihood = {'check_bounds': True,
                     'force_no_add_image': False,
                     'source_marg': False,
                     'image_position_uncertainty': 0.004,
                     'source_position_tolerance': 0.001,
                     'source_position_sigma': 0.001,
                     'prior_lens': [[0, 'e1', 0, 0.2], [0, 'e2', 0, 0.2]],
                     }

multi_band_list = [[kwargs_data, kwargs_psf, kwargs_numerics]]
kwargs_data_joint = {'multi_band_list': multi_band_list, 'multi_band_type': 'multi-linear'}

# initial guess, initial spread, fixed parameters, lower and upper bounds
lens_params = [[{'theta_E': 1.2, 'e1': 0, 'e2': 0, 'gamma': 2., 'center_x': 0., 'center_y': 0}, {'gamma1': 0, 'gamma2': 0}],
               [{'theta_E': 0.3, 'e1': 0.2, 'e2': 0.2, 'gamma': .2, 'center_x': 0.1, 'center_y': 0.1}, {'gamma1': 0.1, 'gamma2': 0.1}],
               [{}, {'ra_0': 0, 'dec_0': 0}],
               [{'theta_E': 0, 'e1': -0.5, 'e2': -0.5, 'gamma': 1.5, 'center_x': -10., 'center_y': -10}, {'gamma1': -0.5, 'gamma2': -0.5}],
               [{'theta_E': 10, 'e1': 0.5, 'e2': 0.5, 'gamma': 2.5, 'center_x': 10., 'center_y': 10}, {'gamma1': 0.5, 'gamma2': 0.5}]]
source_params = [[{'R_sersic': 0.03, 'n_sersic': 1., 'e1': 0, 'e2': 0, 'center_x': 0, 'center_y': 0}],
                 [{'R_sersic': 0.1, 'n_sersic': .5, 'center_x': .1, 'center_y': 0.1, 'e1': 0.2, 'e2': 0.2}],
                 [{}],
                 [{'R_sersic': 0.001, 'n_sersic': .5, 'e1': -0.5, 'e2': -0.5, 'center_x': -10, 'center_y': -10}],
                 [{'R_sersic': 10, 'n_sersic': 5., 'e1': 0.5, 'e2': 0.5, 'center_x': 10, 'center_y': 10}]]
lens_light_params = [[{'R_sersic': 0.1, 'n_sersic': 1, 'e1': 0, 'e2': 0, 'center_x': 0, 'center_y': 0}],
                     [{'R_sersic': 0.1, 'n_sersic': 0.2, 'e1': 0.1, 'e2': 0.1, 'center_x': .1, 'center_y': 0.1}],
                     [{}],
                     [{'R_sersic': 0.001, 'n_sersic': 0.5, 'e1': -0.5, 'e2': -0.5, 'center_x': -10, 'center_y': -10}],
                     [{'R_sersic': 10, 'n_sersic': 5., 'e1': 0.5, 'e2': 0.5, 'center_x': 10, 'center_y': 10}]]
ps_params = [[{'ra_image': x_image + 0.01, 'dec_image': y_image - 0.01}],
             [{'ra_image': [0.02] * 4, 'dec_image': [0.02] * 4}],
             [{}],
             [{'ra_image': -10 * np.ones_like(x_image), 'dec_image': -10 * np.ones_like(y_image)}],
             [{'ra_image': 10 * np.ones_like(x_image), 'dec_image': 10 * np.ones_like(y_image)}]]

kwargs_params = {'lens_model': lens_params,
                 'source_model': source_params,
                 'lens_light_model': lens_light_params,
                 'point_source_model': ps_params}

fitting_seq = FittingSequence(kwargs_data_joint, kwargs_model, kwargs_constraints, kwargs_likelihood, kwargs_params)
fitting_kwargs_list = [['PSO', {'sigma_scale': 1., 'n_particles': 100, 'n_iterations': 200}]]
chain_list = fitting_seq.fit_sequence(fitting_kwargs_list)
kwargs_result = fitting_seq.best_fit()

## How good does the smooth fit look?

The normalized residuals show how well the smooth model reproduces the data. Look near the perturbed image: the perturber also distorts the extended arc around it, which leaves structure in the residuals. Using this kind of arc distortion is another way to find subhalos, called *gravitational imaging*. In real data with lower signal-to-noise it is often hard to see.

Also compare the fitted macro parameters with the true ones printed below. The smooth model partly compensates for the missing perturber by changing its own parameters.

The quasar fluxes are free parameters, so the model matches every image's brightness, even the perturbed one. The anomaly only shows up when we ask whether those fluxes are **consistent with the lens model**.

In [ ]:
modelPlot = ModelPlot(multi_band_list, kwargs_model, kwargs_result)

f, axes = plt.subplots(1, 3, figsize=(18, 5))
modelPlot.data_plot(ax=axes[0])
modelPlot.model_plot(ax=axes[1])
modelPlot.normalized_residual_plot(ax=axes[2], vmin=-6, vmax=6)
plt.show()

print('true macro lens:  ', {k: round(float(v), 3) for k, v in kwargs_lens_macro[0].items()})
print('fitted macro lens:', {k: round(float(v), 3) for k, v in kwargs_result['kwargs_lens'][0].items()})

## Flux-ratio residuals

For each image we compare two numbers:
- the **measured** flux, from the fitted quasar amplitudes;
- the **predicted** flux, from the magnification of the best-fit smooth lens model at the image position.

We don't know the intrinsic quasar brightness, so we only use flux **ratios**, relative to the brightest image. The residual $\frac{(F_i/F_\mathrm{ref})_\mathrm{measured}}{(F_i/F_\mathrm{ref})_\mathrm{predicted}} - 1$ is zero for all images when the smooth model is correct.

For comparison we also show the flux-ratio change the perturber causes relative to the *true* macro model. We can only compute this because we made the simulation.

In [ ]:
kwargs_ps_fit = kwargs_result['kwargs_ps'][0]
x_fit, y_fit = np.array(kwargs_ps_fit['ra_image']), np.array(kwargs_ps_fit['dec_image'])
flux_measured = np.array(kwargs_ps_fit['point_amp'])
mu_predicted = np.abs(lens_model_macro.magnification(x_fit, y_fit, kwargs=kwargs_result['kwargs_lens']))

ref = np.argmax(flux_measured)  # use the brightest image as reference
ratio_measured = flux_measured / flux_measured[ref]
ratio_predicted = mu_predicted / mu_predicted[ref]
residual_fit = ratio_measured / ratio_predicted - 1

# what the perturber did relative to the true (unperturbed) macro model
ratio_true = np.abs(mag_true) / np.abs(mag_true[ref])
ratio_macro_true = np.abs(mag_macro) / np.abs(mag_macro[ref])
residual_true = ratio_true / ratio_macro_true - 1

print(f'reference image: {labels[ref]}')
print('image   measured ratio   smooth-model ratio   residual')
for i in range(4):
    print(f'  {labels[i]}        {ratio_measured[i]:.3f}             {ratio_predicted[i]:.3f}           {residual_fit[i]:+.1%}')

w = 0.38
xs = np.arange(4)
f, ax = plt.subplots(figsize=(8, 5))
ax.axhspan(-0.05, 0.05, color='0.9', label='$\\pm$5% (roughly a good measurement error)')
ax.bar(xs - w / 2, residual_fit, w, label='measured vs. best-fit smooth model')
ax.bar(xs + w / 2, residual_true, w, label='effect of perturber vs. true macro model')
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(xs)
ax.set_xticklabels([f'{labels[i]}\n({"minimum" if mag_macro[i] > 0 else "saddle"})' for i in range(4)])
ax.set_ylabel('flux-ratio residual')
ax.set_title(f'flux ratios relative to image {labels[ref]}; SIS perturber next to image {labels[i_pert]}')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1)
plt.show()

## Things to think about and try

- The smooth model reproduces the image positions exactly (the solver enforces this) and most of the arc, but it can't reproduce the flux of the perturbed image. Why are fluxes so much more sensitive to a small perturber than positions? *(Hint: positions depend on the deflection angle, fluxes on its derivatives.)*
- With the default settings the perturber **demagnifies** the saddle image. Set `perturbed_image = 'minimum'` and `offset_sub = 0.3`, then re-run. What happens to a minimum image? (Closer than about 0.15″ to the minimum image, this perturber splits the image, and the notebook prints a warning.)
- Vary `theta_E_sub` and `offset_sub`. How small can the perturber get before its effect drops below the ~5% measurement error? How close does it have to be?
- In a real lens, microlensing by stars in the lens galaxy also changes fluxes. How could you tell microlensing from a dark matter subhalo? *(Think about the size of the quasar emission region you observe.)*
- **Harder:** add `'SIS'` to the lens model used in the fit (in `kwargs_model` and `kwargs_params`) and check whether the fit recovers the perturber's mass and position.